In [117]:
import yfinance as yf

In [118]:
import numpy as np

In [119]:
import math

In [120]:
import sys, os

In [121]:
sys.path.append(os.path.abspath(".."))

In [122]:
import equitytools_local.data as datatools

In [123]:
from importlib import reload
reload(datatools)

<module 'equitytools_local.data' from 'C:\\Users\\jonsw\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\Scripts\\equitytools_local\\data.py'>

## Stock Option Inputs

In [124]:
ticker_symbol = "MSFT"

In [125]:
ticker = datatools.get_ticker_data(ticker_symbol)

In [126]:
option_type = "call"

In [127]:
strike_price = 400

In [128]:
time_to_expiry = 81 / 365

# Retrieve supporting data


## Fetch 3-month T-bill yield (^IRX)


In [129]:
t_bill = datatools.get_3_month_tbill()

### Get recent historical data for T-bill

In [130]:
hist = t_bill.history(period="5d")  # last 5 days

In [131]:
risk_free_rate = hist['Close'].iloc[-1] / 100  # Convert % to decimal

In [132]:
print(f"Latest 3-month T-bill yield (risk-free rate): {risk_free_rate:.4%}")

Latest 3-month T-bill yield (risk-free rate): 3.5930%


In [133]:
print(risk_free_rate)

0.035929999351501464


In [134]:
time_to_expiry = 81 / 365

In [135]:
current_price = ticker.info['currentPrice']

In [136]:
print(current_price)

401.32


In [137]:
data = datatools.load_60_days_of_prices(ticker_symbol)

[*********************100%***********************]  1 of 1 completed


In [138]:
close_prices = data["Close"]

## Supporting transformations and calculations 

In [139]:
spot_price = close_prices.iat[-1, 0]

In [140]:
returns = np.log(close_prices.MSFT / close_prices.MSFT.shift(1)).dropna()

In [141]:
volatility = np.std(returns) * np.sqrt(252)

In [142]:
d1 = (np.log(spot_price / strike_price) + (risk_free_rate + 0.5 * volatility ** 2) * time_to_expiry) / (volatility * np.sqrt(time_to_expiry))

In [143]:
d2 = d1 - volatility * np.sqrt(time_to_expiry)

In [144]:
import equitytools_local.modeling as modelingtools

In [145]:
reload(modelingtools)

<module 'equitytools_local.modeling' from 'C:\\Users\\jonsw\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\Scripts\\equitytools_local\\modeling.py'>

In [146]:
if option_type == "call":
    option_price = spot_price * np.exp(-0) * 0.5 * (1 + modelingtools.erf(d1 / np.sqrt(2))) - strike_price * np.exp(-risk_free_rate * time_to_expiry) * 0.5 * (1 + modelingtools.erf(d2 / np.sqrt(2)))
    delta = 0.5 * (1 + modelingtools.erf(d1 / np.sqrt(2)))
else:
    option_price = strike_price * np.exp(-risk_free_rate * time_to_expiry) * 0.5 * (1 - modelingtools.erf(d2 / np.sqrt(2))) - spot_price * np.exp(-0) * 0.5 * (1 - modelingtools.erf(d1 / np.sqrt(2)))
    delta = -0.5 * (1 - modelingtools.erf(d1 / np.sqrt(2)))

In [147]:
gamma = np.exp(-d1 ** 2 / 2) / (spot_price * volatility * np.sqrt(2 * np.pi * time_to_expiry))

## Output 1 - Call

In [148]:
print ([option_price, delta, gamma])

[np.float64(26.29665930151313), np.float64(0.559665827545949), np.float64(0.006519275687780777)]


In [149]:
option_type = "put"

In [150]:
if option_type == "call":
    option_price = spot_price * np.exp(-0) * 0.5 * (1 + modelingtools.erf(d1 / np.sqrt(2))) - strike_price * np.exp(-risk_free_rate * time_to_expiry) * 0.5 * (1 + modelingtools.erf(d2 / np.sqrt(2)))
    delta = 0.5 * (1 + modelingtools.erf(d1 / np.sqrt(2)))
else:
    option_price = strike_price * np.exp(-risk_free_rate * time_to_expiry) * 0.5 * (1 - modelingtools.erf(d2 / np.sqrt(2))) - spot_price * np.exp(-0) * 0.5 * (1 - modelingtools.erf(d1 / np.sqrt(2)))
    delta = -0.5 * (1 - modelingtools.erf(d1 / np.sqrt(2)))

In [151]:
gamma = np.exp(-d1 ** 2 / 2) / (spot_price * volatility * np.sqrt(2 * np.pi * time_to_expiry))

## Output 2 - Put

In [152]:
print ([option_price, delta, gamma])

[np.float64(21.799930928888557), np.float64(-0.440334172454051), np.float64(0.006519275687780777)]
